In [154]:
import behaviors
import no_signaling_sets
import numpy as np
import samplers
import scipy
from loguru import logger


In [155]:
verbose = False

delta = 2
m = 2

In [156]:
import scipy.optimize


def test_for_srns(sample: behaviors.RoutedBehavior):
    srns_set = no_signaling_sets.ShortRangeNoSignalingSet(
        delta=delta,
        m=m,
        measured_behavior=sample,
        )

    A_eq, b_eq = srns_set.get_equations(measured_behavior=sample)

    q_shape = behaviors.LatentSRNSBehavior(
        delta=delta,
        m=m,
    ).vector_shape[0]

    lb = np.zeros(q_shape+1)
    rb = np.ones(q_shape+1)
    rb[0] = 2

    bounds = [(lb[i], rb[i]) for i in range(q_shape+1)]

    c = np.zeros(q_shape+1)
    c[0] = -1

    res:scipy.optimize.OptimizeResult = scipy.optimize.linprog(
        c=c,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
    )

    return res


In [157]:
sampler = samplers.NoSignalingSampler(delta, m)

sampled_behavior = sampler.sample()

srns_set = no_signaling_sets.ShortRangeNoSignalingSet(
    delta=delta,
    m=m,
    measured_behavior=sampled_behavior,
)

A_eq, b_eq = srns_set.get_equations(measured_behavior=sampled_behavior)

q_shape = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
).vector_shape[0]

lb = np.zeros(q_shape+1)
rb = np.ones(q_shape+1)
rb[0] = 2

bounds = [(lb[i], rb[i]) for i in range(q_shape+1)]

c = np.zeros(q_shape+1)
c[0] = -1

if verbose:
    with np.printoptions(threshold=np.inf):
        print("A_eq")
        print(A_eq)
        print("b_eq")
        print(b_eq)
        print("bounds")
        print(bounds)
        print("c")
        print(c)
        print('\n\n---------\n\n')

print(f"Sampled behavior : {sampled_behavior}")
print(f"Sampled behavior is tested [{sampled_behavior.is_no_signaling()}] to being no signaling")


2025-05-19 17:31:33.538 | SUCCESS  | samplers:sample_multiple:100 - Samples shape: (1, 32)
2025-05-19 17:31:33.539 | DEBUG    | no_signaling_sets:express_as_function_of_q:189 - Estimated memory complexity of lacking_betas: 56 bytes


Sampled behavior : Behavior:
Short path (z=S):
[[0.05936121 0.11018712 0.05667876 0.02368534]
 [0.11856308 0.06773717 0.33516942 0.36816284]
 [0.39289164 0.19441532 0.39557408 0.2809171 ]
 [0.42918407 0.62766039 0.21257773 0.32723472]]
Long path (z=L) :
[[0.05868825 0.0015291  0.27999227 0.18466753]
 [0.11923605 0.1763952  0.11185591 0.20718066]
 [0.48780563 0.63311161 0.2665016  0.44997318]
 [0.33427008 0.18896409 0.34165021 0.15817863]]
------------
Sampled behavior is tested [True] to being no signaling


In [158]:
res:scipy.optimize.OptimizeResult = scipy.optimize.linprog(
    c=c,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
)

print(res.success)
print(-res.fun)
print(res.status)
print(res.x)

True
1.006154034978908
0
[1.00615403 0.05818801 0.10932671 0.05548906 0.0222926  0.11775422
 0.06661552 0.33569356 0.36889002 0.393771   0.19407325 0.39646995
 0.28110736 0.43028677 0.62998452 0.21234743 0.32771002 0.
 0.0732597  0.05751091 0.20691715 0.         0.11100577 0.11843132
 0.         0.48926909 0.10898959 0.         0.15761356 0.1462002
 0.34221423 0.18858847 0.        ]


In [159]:
yielded_behavior = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
    vector=np.clip(np.array(res.x[1:]), 0, 1),
)

test_tolerance = 1e-10

print(f"Yielded behavior is {yielded_behavior}")
print(f"Yielded behavior is tested [{yielded_behavior.is_no_signaling(atol=test_tolerance)}] to being no signaling")


Yielded behavior is Behavior:
Short path (z=S):
[[0.05818801 0.10932671 0.05548906 0.0222926 ]
 [0.11775422 0.06661552 0.33569356 0.36889002]
 [0.393771   0.19407325 0.39646995 0.28110736]
 [0.43028677 0.62998452 0.21234743 0.32771002]]
Long path (z=L) :
[[0.         0.0732597 ]
 [0.05751091 0.20691715]
 [0.         0.11100577]
 [0.11843132 0.        ]
 [0.48926909 0.10898959]
 [0.         0.15761356]
 [0.1462002  0.34221423]
 [0.18858847 0.        ]]
------------
Yielded behavior is tested [True] to being no signaling


In [160]:
sanity_check = A_eq[:, 1:] @ yielded_behavior.get_vector()
sanity_check = sanity_check[:2*delta**2*m**2]

In [161]:
sanity_check = behaviors.RoutedBehavior(
    delta=delta,
    m=m,
    vector=sanity_check,
)
print(f"Sanity check is {sanity_check}")
print(f"Sanity check is tested [{sanity_check.is_no_signaling()}] to being no signaling")

Sanity check is Behavior:
Short path (z=S):
[[0.05818801 0.10932671 0.05548906 0.0222926 ]
 [0.11775422 0.06661552 0.33569356 0.36889002]
 [0.393771   0.19407325 0.39646995 0.28110736]
 [0.43028677 0.62998452 0.21234743 0.32771002]]
Long path (z=L) :
[[0.05751091 0.         0.28017685 0.18426547]
 [0.11843132 0.17594223 0.11100577 0.20691715]
 [0.48926909 0.6354693  0.26660315 0.45120383]
 [0.33478868 0.18858847 0.34221423 0.15761356]]
------------
Sanity check is tested [True] to being no signaling


In [162]:
res = test_for_srns(sampled_behavior)

print(res.success)
print(-res.fun)
print(res.status)
print(res.x)

2025-05-19 17:31:33.589 | DEBUG    | no_signaling_sets:express_as_function_of_q:189 - Estimated memory complexity of lacking_betas: 56 bytes


True
1.006154034978908
0
[1.00615403 0.05818801 0.10932671 0.05548906 0.0222926  0.11775422
 0.06661552 0.33569356 0.36889002 0.393771   0.19407325 0.39646995
 0.28110736 0.43028677 0.62998452 0.21234743 0.32771002 0.
 0.0732597  0.05751091 0.20691715 0.         0.11100577 0.11843132
 0.         0.48926909 0.10898959 0.         0.15761356 0.1462002
 0.34221423 0.18858847 0.        ]


In [166]:
res = test_for_srns(behaviors.pr_box)

print(res.success)
print(-res.fun)
print(res.status)
print(res.x)

pr_match = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
    vector=np.clip(np.array(res.x[1:]), 0, 1),
)
print(f"PR match is {pr_match}")
print(f"PR match is tested [{pr_match.is_no_signaling()}] to being no signaling")

pr_box_check = A_eq[:, 1:] @ pr_match.get_vector()
pr_box_check = pr_box_check[:2*delta**2*m**2]
pr_box_check = behaviors.RoutedBehavior(
    delta=delta,
    m=m,
    vector=pr_box_check,
)
print(f"PR box check is {pr_box_check}")
print(f"PR box check is tested [{pr_box_check.is_no_signaling()}] to being no signaling")

2025-05-19 17:38:33.290 | DEBUG    | no_signaling_sets:express_as_function_of_q:189 - Estimated memory complexity of lacking_betas: 56 bytes


True
1.0
0
[ 1.   0.5  0.5  0.5  0.  -0.  -0.  -0.   0.5 -0.  -0.  -0.   0.5  0.5
  0.5  0.5 -0.   0.5  0.  -0.   0.5  0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.5  0.5 -0. ]
PR match is Behavior:
Short path (z=S):
[[ 0.5  0.5  0.5  0. ]
 [-0.  -0.  -0.   0.5]
 [-0.  -0.  -0.   0.5]
 [ 0.5  0.5  0.5 -0. ]]
Long path (z=L) :
[[ 0.5  0. ]
 [-0.   0.5]
 [ 0.   0. ]
 [ 0.   0. ]
 [ 0.   0. ]
 [ 0.   0. ]
 [ 0.   0.5]
 [ 0.5 -0. ]]
------------
PR match is tested [True] to being no signaling
PR box check is Behavior:
Short path (z=S):
[[0.5 0.5 0.5 0. ]
 [0.  0.  0.  0.5]
 [0.  0.  0.  0.5]
 [0.5 0.5 0.5 0. ]]
Long path (z=L) :
[[0.5 0.5 0.5 0. ]
 [0.  0.  0.  0.5]
 [0.  0.  0.  0.5]
 [0.5 0.5 0.5 0. ]]
------------
PR box check is tested [True] to being no signaling


### Loop over samples to search for nontrivial alphas

In [163]:
# counter = 0
# logger.remove()

# while res.fun == 0:
#     sampled_behavior = sampler.sample()
#     srns_set = no_signaling_sets.ShortRangeNoSignalingSet(
#         delta=delta,
#         m=m,
#         measured_behavior=sampled_behavior,
#     )
#     A_eq, b_eq = srns_set.get_equations(measured_behavior=sampled_behavior)
#     q_shape = behaviors.LatentSRNSBehavior(
#         delta=delta,
#         m=m,
#     ).vector_shape[0]
#     lb = np.zeros(q_shape+1)
#     rb = np.ones(q_shape+1)
#     rb[0] = 2
#     bounds = [(lb[i], rb[i]) for i in range(q_shape+1)]
#     c = np.zeros(q_shape+1)
#     c[0] = 1

#     res:scipy.optimize.OptimizeResult = scipy.optimize.linprog(
#         c=c,
#         A_eq=A_eq,
#         b_eq=b_eq,
#         bounds=bounds,
#     )

#     counter += 1
#     if counter % 100 == 0:
#         print(f"Counter: {counter}")